# Week 3 — Embeddings + Indexing
Embed all text chunks (Gemini) and images (CLIP), build FAISS indexes.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/multimodal-rag-enterprise-km

Mounted at /content/drive
/content/drive/MyDrive/multimodal-rag-enterprise-km


In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.3 MB/s eta 0:00:00


In [4]:
import sys, os, json
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')

from src.embeddings import embed_all_text_chunks, embed_all_images
from src.indexing import build_and_save_all

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
with open('data/processed/text_chunks/chunks.json') as f:
    data = json.load(f)
text_chunks = data['text_chunks']
images = data['images']
print(len(text_chunks), 'text chunks,', len(images), 'images')

73 text chunks, 137 images


In [6]:
from google.colab import userdata
key = userdata.get("GEMINI_API_KEY")
print("Key found:", bool(key), "| length:", len(key) if key else 0)

Key found: True | length: 53


In [7]:
# Cell 1
import os
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
assert api_key, "Secret not found — check the key icon panel and notebook access toggle"
os.environ["GEMINI_API_KEY"] = api_key

import google.generativeai as genai
genai.configure(api_key=api_key)
print("Configured OK")

Configured OK


In [8]:
# Cell 2 — import src AFTER configuring, in the same fresh session
import sys
sys.path.append('/content/drive/MyDrive/multimodal-rag-enterprise-km')

from src.embeddings import embed_all_text_chunks, embed_all_images

In [9]:
# Cell 3
text_vectors = embed_all_text_chunks(text_chunks)
image_vectors = embed_all_images(images)
print(text_vectors.shape, image_vectors.shape)

  embedded 10/73 text chunks
  embedded 20/73 text chunks
  embedded 30/73 text chunks
  embedded 40/73 text chunks
  embedded 50/73 text chunks
  embedded 60/73 text chunks
  embedded 70/73 text chunks
Loading CLIP model (first call only)...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

  embedded 10/137 images
  embedded 20/137 images
  embedded 30/137 images
  embedded 40/137 images
  embedded 50/137 images
  embedded 60/137 images
  embedded 70/137 images
  embedded 80/137 images
  embedded 90/137 images
  embedded 100/137 images
  embedded 110/137 images
  embedded 120/137 images
  embedded 130/137 images
(73, 3072) (137, 512)


In [10]:
text_index, image_index = build_and_save_all(text_chunks, text_vectors, images, image_vectors, out_dir='../indexes')

Saved indexes and metadata to ../indexes
